# Vehicle Breakdown Prediction with KNN

This notebook documents the end-to-end process for building and optimizing a machine learning model to predict vehicle breakdowns, focusing exclusively on the **K-Nearest Neighbors (KNN)** algorithm.

The workflow covers:

1. **Data Processing:** Cleaning and preparing raw service order data.
2. **Outlier Treatment:** Handling extreme values to create a more robust model.
3. **Feature Engineering:** Creating insightful features from historical data, including advanced metrics based on usage (odometer) and service history.
4. **Target Definition:** Properly defining the prediction target (`breakdown`), avoiding data leakage.
5. **Class Imbalance Treatment:** Using SMOTE to balance the training data.
6. **Baseline Model Evaluation:** Analyzing a KNN model with default parameters to establish a baseline.
7. **Hyperparameter Optimization:** Fine-tuning the KNN model to maximize its predictive performance.

## 1. Setup and Dependencies

The first step is to set up the environment, installing and importing all necessary libraries for the project. The most important package here is `imbalanced-learn`, which will give us access to the SMOTE technique.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

## 2. Processing, Cleaning and Outlier Treatment

In this phase, we load the raw data, perform essential cleaning, and treat outliers. This includes removing irrelevant columns, standardizing formats, converting textual data to numerical, and limiting extreme values that could distort the model.

**Important:** To run this cell, you need to upload your `SERVICE_ORDER_BASE.xlsx` file to the Colab environment.

In [ ]:
# ==============================================================================
# STEP 1: DATA PROCESSING AND CLEANING
# ==============================================================================
print("="*20)
print("STEP 1: DATA PROCESSING AND CLEANING")
print("="*20)

input_file = "data/SERVICE_ORDER_BASE.xlsx"
output_file = "data/SERVICE_ORDER_CLEAN.xlsx"
print("Starting data processing...")
start_time = time.time()

try:
    df = pd.read_excel(input_file)
    to_remove = [
        "MODEL TYPE DESCRIPTION", "ASSET PURCHASE DATE", "ITEM OF LEDGER ACCOUNT",
        "LEDGER ACCOUNT DESCRIPTION", "MAINTENANCE TYPE", "SERVICE ORDER", "INVOICE",
        "SUPPLIER'S CODE", "SUPPLIER'S STORE", "NAME OR COMPANY NAME"
    ]
    df = df.drop(columns=[c for c in to_remove if c in df.columns], errors="ignore")

    if "COUNTER  OF SERVICE ORDER" in df:
        df = df.rename(columns={"COUNTER  OF SERVICE ORDER": "ODOMETER"})
    if "SERVICE ORDER ORIGINAL DATE" in df:
        df["SERVICE ORDER ORIGINAL DATE"] = (
            df["SERVICE ORDER ORIGINAL DATE"].astype(str)
            .str.strip(" '\"\t")
            .str.replace(r"[^\d/]", "", regex=True)
        )
        def format_date(d):
            if d.isdigit() and len(d) == 8: return f"{d[6:]}/{d[4:6]}/{d[:4]}"
            return d
        df["SERVICE ORDER ORIGINAL DATE"] = df["SERVICE ORDER ORIGINAL DATE"].apply(format_date)

    if "TIER" in df:
        df["TIER"] = df["TIER"].replace({"TIER 1": 1, "T1": 1, "TIER 2": 2, "T2": 2})
    if "ASSET STATUS" in df:
        df["ASSET STATUS"] = df["ASSET STATUS"].map({"ACTIVE": 1, "INACTIVE": 0}).fillna(df["ASSET STATUS"])
    if "PREVENTIVE_CORRECTIVE MAINTENANCE" in df:
        df["PREVENTIVE_CORRECTIVE MAINTENANCE"] = df["PREVENTIVE_CORRECTIVE MAINTENANCE"].map({"PREVENTIVE": 1, "CORRECTIVE": 0}).fillna(df["PREVENTIVE_CORRECTIVE MAINTENANCE"])

    req = ["PRODUCT QUANTITY", "UNIT VALUE", "GRAND TOTAL"]
    if all(c in df for c in req):
        df = df.dropna(subset=req)

    if "SERVICE ORDER ORIGINAL DATE" in df:
        df["SERVICE ORDER ORIGINAL DATE"] = pd.to_datetime(df["SERVICE ORDER ORIGINAL DATE"], dayfirst=True, errors="coerce")
        df["SERVICE_ORDER_year"] = df["SERVICE ORDER ORIGINAL DATE"].dt.year
        df["SERVICE_ORDER_month"] = df["SERVICE ORDER ORIGINAL DATE"].dt.month
        df["SERVICE_ORDER_day"] = df["SERVICE ORDER ORIGINAL DATE"].dt.day
        df = df.drop(columns=["SERVICE ORDER ORIGINAL DATE"])

    le = LabelEncoder()
    for col in ["MODEL TYPE CODE", "PRODUCT CODE", "ASSET CODE"]:
        if col in df:
            df[col + "_enc"] = le.fit_transform(df[col].astype(str))
            df = df.drop(columns=[col])

    df.to_excel(output_file, index=False)
    print("Data processed in", round(time.time() - start_time, 2), "s")

    # ==============================================================================
    # STEP 2: OUTLIER ANALYSIS AND TREATMENT
    # ==============================================================================
    print("\n" + "="*20)
    print("STEP 2: OUTLIER ANALYSIS AND TREATMENT")
    print("="*20)

    cols = ["GRAND TOTAL", "PRODUCT QUANTITY", "UNIT VALUE", "ODOMETER"]
    for c in cols:
        if c in df:
            data = df[c].dropna()
            Q1, Q3 = data.quantile([0.25, 0.75])
            IQR = Q3 - Q1
            low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            print(f"{c}: mean={data.mean():.2f}, outliers={((data<low)|(data>high)).sum()}")
            df[c] = df[c].clip(low, high)
    out_file = "data/SERVICE_ORDER_BASE_outliers_treated.xlsx"
    df.to_excel(out_file, index=False)
    print("Outliers treated and saved.")

except FileNotFoundError:
    print(f"ERROR: File '{input_file}' not found. Please check if the file has been uploaded to the Colab environment.")
    df = None # Ensures 'df' will not be used if there's an error
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df = None

## 3. Dataset Preparation for Modeling

This is the most complex step, where the data is transformed into an ideal format for KNN training. It includes advanced feature creation, correct target definition to avoid *data leakage*, data splitting, balancing with SMOTE, and scaling.

In [ ]:
print("\n" + "="*20)
print("STEP 3: DATASET PREPARATION FOR MODELING")
print("="*20)

try:
    # Ensures the previous cell was executed successfully
    if 'df' in locals() and df is not None:
        if all(c in df.columns for c in ["SERVICE_ORDER_year", "SERVICE_ORDER_month", "SERVICE_ORDER_day"]):
            df["SERVICE_ORDER_date"] = pd.to_datetime(
                dict(year=df["SERVICE_ORDER_year"], month=df["SERVICE_ORDER_month"], day=df["SERVICE_ORDER_day"]),
                errors="coerce"
            )
        else:
            raise ValueError("Valid date columns not found!")

        vid = "ASSET CODE_enc" if "ASSET CODE_enc" in df.columns else "ASSET CODE"
        if vid not in df.columns or "ODOMETER" not in df.columns:
            raise ValueError("Vehicle identification column or odometer not found!")

        df = df.dropna(subset=["GRAND TOTAL", "SERVICE_ORDER_date", "ODOMETER"])
        df = df.sort_values([vid, "SERVICE_ORDER_date"])

        feats = []
        for v in df[vid].unique():
            vdf = df[df[vid] == v]
            for _, row in vdf.iterrows():
                date = row["SERVICE_ORDER_date"]
                prev = vdf[vdf["SERVICE_ORDER_date"] < date]
                days = (date - prev["SERVICE_ORDER_date"].max()).days if len(prev) else 0
                feats.append({
                    "vehicle_id": v, "date": date, "cost": row["GRAND TOTAL"],
                    "days_since_last": days, "odometer": row["ODOMETER"]
                })
        featdf = pd.DataFrame(feats)

        threshold_cost = featdf["cost"].quantile(0.75)
        threshold_days = 180
        featdf["is_high_cost"] = (featdf["cost"] > threshold_cost).astype(int)
        featdf["is_long_interval"] = (featdf["days_since_last"] > threshold_days).astype(int)
        featdf = featdf.sort_values(["vehicle_id", "date"])
        featdf["breakdown"] = featdf.groupby("vehicle_id")[["is_high_cost", "is_long_interval"]].shift(-1).any(axis=1)
        featdf = featdf.dropna(subset=["breakdown"])
        featdf["breakdown"] = featdf["breakdown"].astype(int)

        if featdf["breakdown"].nunique() < 2:
            raise ValueError("The target variable has no variation (only one class present).")

        print("Creating advanced features...")
        featdf["rolling_cost_mean"] = featdf.groupby("vehicle_id")["cost"].transform(lambda x: x.rolling(3, min_periods=1).mean())
        featdf['service_count'] = featdf.groupby('vehicle_id').cumcount()
        featdf['km_since_last'] = featdf.groupby('vehicle_id')['odometer'].diff().clip(lower=0)
        featdf['km_per_day'] = (featdf['km_since_last'] / featdf['days_since_last']).replace([np.inf, -np.inf], 0)
        featdf['avg_cost_so_far'] = featdf.groupby('vehicle_id')['cost'].transform(lambda x: x.shift(1).expanding().mean())
        featdf['std_dev_cost_so_far'] = featdf.groupby('vehicle_id')['cost'].transform(lambda x: x.shift(1).expanding().std())
        featdf['avg_days_between_services'] = featdf.groupby('vehicle_id')['days_since_last'].transform(lambda x: x.shift(1).expanding().mean())
        featdf['days_overdue'] = featdf['days_since_last'] - featdf['avg_days_between_services']
        featdf = featdf.fillna(0)
        print("Features created.")

        featdf = featdf.sort_values("date")
        split = int(len(featdf) * 0.8)
        train, test = featdf.iloc[:split], featdf.iloc[split:]

        feature_names = [
            "cost", "days_since_last", "rolling_cost_mean", "service_count",
            "km_since_last", "km_per_day", "avg_cost_so_far",
            "std_dev_cost_so_far", "avg_days_between_services", "days_overdue"
        ]
        Xtr, ytr = train[feature_names], train["breakdown"]
        Xte, yte = test[feature_names], test["breakdown"]

        print("\nApplying SMOTE to training data...")
        smote = SMOTE(random_state=42)
        Xtr_resampled, ytr_resampled = smote.fit_resample(Xtr, ytr)

        print("\nScaling features...")
        scaler = StandardScaler()
        Xtr_scaled_resampled = scaler.fit_transform(Xtr_resampled)
        Xte_scaled = scaler.transform(Xte)
    else:
        print("Data preparation failed in previous steps. The script cannot continue.")
except Exception as e:
    print(f"An unexpected error occurred during dataset preparation: {e}")

## 4. Baseline KNN Model Evaluation

After data preparation, we train a first KNN model with default configuration (`n_neighbors=5`). The evaluation of this model provides us with a **baseline**, a reference point to understand the initial performance before optimization.

In [ ]:
print("\n" + "="*20)
print("STEP 4: BASELINE KNN MODEL EVALUATION")
print("="*20)

try:
    knn_base = KNeighborsClassifier(n_neighbors=5)
    knn_base.fit(Xtr_scaled_resampled, ytr_resampled)
    ypred_knn_base = knn_base.predict(Xte_scaled)

    print("\n[Baseline K-Nearest Neighbors (KNN) Evaluation]")
    print(classification_report(yte, ypred_knn_base))
except NameError:
    print("Training/test data was not defined. Execute the previous cell successfully.")
except Exception as e:
    print(f"An unexpected error occurred during baseline model evaluation: {e}")

### Baseline Model Results Analysis
The baseline model works as an **initial screening network**.
- **Recall (~0.59):** It is able to identify about 59% of actual breakdowns, showing a moderate ability to "capture" problems.
- **Precision (~0.35):** However, when it predicts a breakdown, it is correct only 35% of the time, generating a considerable number of false alarms.
- **F1-Score (~0.44):** The low F1-score indicates a weak balance between the two metrics.

This result shows there is room for improvement, which we will attempt to achieve through optimization.

## 5. Hyperparameter Optimization for KNN

In this final step, we search for the best combination of settings for the KNN model. We use the `RandomizedSearchCV` technique to automatically test 50 different parameter combinations, with the goal of finding the combination that maximizes the **f1-score**.

In [ ]:
print("\n" + "="*20)
print("STEP 5: HYPERPARAMETER OPTIMIZATION FOR KNN")
print("="*20)

try:
    param_grid = {
        'n_neighbors': list(range(3, 31, 2)),
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    }

    knn_opt = KNeighborsClassifier()
    knn_random = RandomizedSearchCV(estimator=knn_opt, param_distributions=param_grid,
                                    n_iter=50, cv=3, verbose=2, random_state=42,
                                    n_jobs=-1, scoring='f1')

    print("\nFitting the model... (This may take several minutes)")
    knn_random.fit(Xtr_scaled_resampled, ytr_resampled)

    print("\nBest parameters found:")
    print(knn_random.best_params_)

    best_model = knn_random.best_estimator_
    ypred_best = best_model.predict(Xte_scaled)

    print("\n[Optimized K-Nearest Neighbors (KNN) Evaluation]")
    print(classification_report(yte, ypred_best))

except NameError:
    print("Training/test data was not defined. Execute the previous cells successfully.")
except Exception as e:
    print(f"An unexpected error occurred during optimization: {e}")

## 6. Final Conclusion

The journey of developing this model, from initial cleaning to hyperparameter optimization, leads us to a clear and fundamental conclusion in data science: **the quality of data and feature engineering surpasses algorithm fine-tuning.**

The optimization confirmed that the baseline model already operated at its "performance plateau", as there were no significant gains when adjusting its internal parameters. This validates that the true performance leaps were obtained in previous steps, specifically through:

1. **Feature Engineering:** Creating variables such as `km_since_last` and `days_overdue` gave the model the necessary context to understand each vehicle's behavior.
2. **Balancing with SMOTE:** The technique allowed the model to learn the patterns of the minority class (the "breakdowns"), significantly increasing its detection capability (recall).

The final result is an optimized KNN model that works as a **screening tool**. It can be used with confidence that it is well configured, presenting the following performance profile:
* **Detection Capability (Recall):** Identifies about **58%** of actual problems.
* **Reliability (Precision):** When it signals a problem, it is correct **35%** of the time.

For future improvements, the most promising path would not be to continue adjusting the KNN, but rather to explore new data sources or test more complex algorithms (such as tree-based models) that can capture more subtle relationships in the features already created.